# Microsoft Agent Framework — Azure OpenAI (Responses API)

В този примерен код ще използвате **Microsoft Agent Framework (MAF)** за създаване на прост агент, поддържан от **Azure OpenAI** с помощта на **Responses API**.

> **Бележка за миграция:** Този пример преди това използваше Semantic Kernel с GitHub Models. Той е прехвърлен към Microsoft Agent Framework, а GitHub Models (оттеглящи се, с прекратяване през юли 2026 г.) са заменени с Azure OpenAI, който поддържа Responses API. `OpenAIChatClient` в MAF се насочва към стабилната крайна точка на Azure OpenAI `/openai/v1/` и по подразбиране използва Responses API.

Целта на този пример е да демонстрира стъпките, които по-късно ще бъдат приложени в допълнителни примерни кодове при реализиране на различни агентни модели.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## Импортиране на необходимите Python пакети


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Дефиниране на инструмент

В Microsoft Agent Framework, **инструмент** е обикновена Python функция, декорирана с `@tool`, която агентът може да извика. По-долу дефинираме инструмент, който връща случайна дестинация за ваканция и избягва повтаряне на предишната.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Създаване на агента

Тук създаваме агента на име `TravelAgent`.

В този пример използваме много основни инструкции. Чувствайте се свободни да променяте тези инструкции, за да наблюдавате как се променя поведението на агента.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Стартиране на агента

Сега можем да стартираме агента. Създаваме `AgentSession`, така че агентът да помни разговора през различните ходове, след което изпращаме два `user_inputs`. Първият иска пътуване; вторият казва, че потребителят не харесва предложението и иска друго — агентът използва историята на сесията плюс инструмента `get_random_destination`, за да отговори.

Можете да променяте тези съобщения, за да наблюдавате как агентът реагира по различен начин. Отговорите се **излъчват на потоци** токен по токен.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
